# Trace-or-Fake: which mechanisms make honest disclosure an equilibrium?

COMSCI/ECON 206 - Computational Microeconomics - Autumn 2026 Session 1

Zhengjun He - Duke Kunshan University

**Q2 (computation) of the PS1 proposal.** What trace-file format and verifier make a per-claim declaration checkable at near-zero cost, without trusting the producer's own record?

This notebook shares one payoff law with the interactive Hugging Face game, so the paper, the notebook and the demo cannot disagree.

---

## Two corrections found by running this notebook

**Correction 1 (sign error).** The deployed game originally computed a long-run score by **adding** a cost index, a benefit index and a stability index. Cost and benefit do not share a sign convention: a higher cost is **worse**, a higher benefit is **better**. Adding them scored the honest action as better on cost (105 vs 80) *and* better on benefit (120 vs 80), so honesty won twice before any rule applied. The spot-check sweep then returned `p* = 0`: faking was never attractive, so the rule question could not be posed at all.

**Correction 2 (honesty was a cost, not an asset).** Once the sign error was fixed, honest and faked traces still earned the **same** base benefit. Honesty was therefore never chosen because it was worth more, only because cheating was punished. That is a tax, not a signal. This notebook adds a market premium `delta`, paid only for a verifiable trace file, so honesty can win on value rather than on deterrence.

Correction 2 was prompted by the certification-design result of Mäkimattila, Shang and Shirakawa (ACM EC 2025), where a sender's **choice** among offered tests is itself informative, and by Spence's (1973) single-crossing condition.

In [ ]:
# No API key, no paid service, CPU only, standard library only.

BASE = 100.0
STABILITY_BASE = 100.0

# A developer publishing a tool chooses a trace-file action.
ACTIONS = {
    #               production cost, base market benefit
    "A_honest":   {"cost":  5.0,  "benefit": 20.0},  # pays to produce a true trace file
    "B_none":     {"cost":  0.0,  "benefit":  0.0},  # no trace file (status quo)
    "C_fake":     {"cost": -20.0, "benefit": 20.0},  # cheap to fake, paid like honest
}
VERIFIER = {"cost": 5.0, "stability": 20.0}

# Two mechanisms:
#   p, F   -> expected fine pF, charged only to a detected fake
#   delta  -> premium, paid only for a verifiable trace file
NO_RULES = {"p": 0.0, "F": 0.0}

for a, v in ACTIONS.items():
    print(f"{a:<10} cost {v['cost']:+6.1f}   base benefit {v['benefit']:+6.1f}")
print(f"\nverifier enrolment: cost +{VERIFIER['cost']}, stability +{VERIFIER['stability']}")

## 1. The payoff function

Net payoff for one cell is benefit plus the premium if honest, minus cost, plus stability if enrolled, minus the expected fine if faking.

The premium falls **only** on the honest action. The expected fine falls **only** on the fake. Those two terms are the entire economic content of the proposal, and they are the two mechanisms a platform can influence.

In [ ]:
def payoff(action, enrolled, p=0.0, F=0.0, delta=0.0, actions=ACTIONS):
    """Net payoff for one (action, verifier) cell.

    Benefit minus cost, plus stability when enrolled, plus the premium delta
    if the action is honest, minus the expected fine pF if the action is a fake.
    """
    a = actions[action]
    net = (BASE + a["benefit"]) - (BASE + a["cost"])
    net += STABILITY_BASE + (VERIFIER["stability"] if enrolled else 0.0)
    if action == "A_honest":
        net += delta
    if action == "C_fake":
        net -= p * F
    return net


ACTION_ORDER = ("A_honest", "B_none", "C_fake")


def payoff_table(p=0.0, F=0.0, delta=0.0, actions=ACTIONS):
    return [{"action": a, "verifier": e,
             "net": payoff(a, e, p, F, delta, actions)}
            for a in ACTION_ORDER for e in (False, True)]


def best_response(p=0.0, F=0.0, delta=0.0):
    rows = payoff_table(p, F, delta)
    top = max(r["net"] for r in rows)
    return [r for r in rows if r["net"] == top]


print("Payoff with NO rules and NO premium (p=0, F=0, delta=0)")
print(f"{'action':<12}{'not enrolled':>14}{'enrolled':>12}")
print("-" * 40)
for a in ACTION_ORDER:
    print(f"{a:<12}{payoff(a, False):>14.0f}{payoff(a, True):>12.0f}")

w0 = best_response()
print("\nBest response: " + " | ".join(f"{r['action']}" for r in w0))
print("The FAKE wins. That is the market failure the proposal is about.")

## 2. Verification checks

In [ ]:
def run_checks():
    c = {}
    # (1) No rules, no premium: the fake wins. The failure exists.
    c["no_rules_fake_wins"] = any(r["action"] == "C_fake" for r in best_response())
    # (2) Strong enforcement alone: honesty is unique.
    w = best_response(p=1.0, F=100.0)
    c["enforcement_makes_honesty_unique"] = (
        len(w) == 1 and w[0]["action"] == "A_honest" and w[0]["verifier"])
    # (3) Premium alone, no enforcement at all: honesty is unique.
    w = best_response(delta=30.0)
    c["premium_alone_makes_honesty_unique"] = (
        len(w) == 1 and w[0]["action"] == "A_honest" and w[0]["verifier"])
    # (4) Status quo cell is fixed by construction.
    sq = [r for r in payoff_table() if r["action"] == "B_none" and not r["verifier"]][0]
    c["status_quo_unchanged"] = sq["net"] == STABILITY_BASE
    # (5) Enrolment is weakly dominant.
    c["enrolment_weakly_dominant"] = all(
        payoff(a, True) > payoff(a, False) for a in ACTION_ORDER)
    # (6) Six cells.
    c["six_cells"] = len(payoff_table()) == 6
    return c


checks = run_checks()
for k, v in checks.items():
    print(f"{'PASS' if v else 'FAIL'}  {k}")
assert all(checks.values()), "a verification check failed"
print("\nAll checks passed.")

## 3. Mechanism 1: the fine (stick only)

Hold the premium at zero and sweep the spot-check rate.

In [ ]:
F_FIXED = 50.0
print(f"Fine sweep, delta = 0, F fixed at {F_FIXED}")
print(f"{'p':>6}  {'best response':<30}{'net':>8}")
print("-" * 46)
for i in range(0, 21):
    p = i / 20
    w = best_response(p=p, F=F_FIXED)
    d = " | ".join(r["action"] for r in w)
    print(f"{p:6.2f}  {d:<30}{w[0]['net']:8.1f}")


def honesty_threshold(F, delta=0.0, steps=200001):
    """Smallest p at which honest is the UNIQUE best response."""
    for i in range(steps):
        p = i / (steps - 1)
        w = best_response(p=p, F=F, delta=delta)
        if len(w) == 1 and w[0]["action"] == "A_honest" and w[0]["verifier"]:
            return p
    return None


p_star = honesty_threshold(F_FIXED)
fake_adv = payoff("C_fake", True) - payoff("A_honest", True)
print(f"\nThreshold p* = {p_star:.4f}  (F = {F_FIXED}, delta = 0)")
print(f"Fake's advantage at p=0: {fake_adv:+.0f}")
print("\nThreshold p* as a function of the fine F:")
print(f"{'F':>7}{'p*':>10}{'p*F':>9}")
print("-" * 26)
for Ft in [25.0, 50.0, 100.0, 200.0, 400.0]:
    ps = honesty_threshold(Ft)
    if ps is None:
        print(f"{Ft:7.0f}{'none':>10}{'n/a':>9}   (F too small: even p=1 cannot deter)")
    else:
        print(f"{Ft:7.0f}{ps:10.4f}{ps * Ft:9.2f}")

## 4. Mechanism 2: the premium (carrot only)

Now switch **off** all enforcement (`p = F = 0`) and sweep the premium. With no fine at all, can honesty survive on value alone?

In [ ]:
print("Premium sweep with NO enforcement (p = 0, F = 0)")
print(f"{'delta':>7}  {'best response':<30}{'net':>8}")
print("-" * 47)
for d in [0.0, 10.0, 20.0, 25.0, 26.0, 30.0, 50.0, 100.0]:
    w = best_response(delta=d)
    s = " | ".join(r["action"] for r in w)
    print(f"{d:7.0f}  {s:<30}{w[0]['net']:8.1f}")

print()
print("Payoff by action, with and without a premium:")
print(f"{'action':<12}{'delta=0':>10}{'delta=30':>10}")
print("-" * 32)
for a in ACTION_ORDER:
    print(f"{a:<12}{payoff(a, True, delta=0.0):>10.0f}{payoff(a, True, delta=30.0):>10.0f}")

w_no = best_response(delta=0.0)
w_yes = best_response(delta=30.0)
print(f"\ndelta=0  -> winner: {' | '.join(r['action'] for r in w_no)}")
print(f"delta=30 -> winner: {' | '.join(r['action'] for r in w_yes)}")
print("\nWith a premium, honesty wins with NO enforcement at all.")

## 5. The two mechanisms are substitutes

The fine and the premium act on the same gap, so they trade off exactly: the fake's cost advantage is 25, and honesty needs the premium plus the expected fine to exceed it.

A platform can buy deterrence with either more checking (costly to run) or a larger premium (costly to create).

In [ ]:
print("Combined threshold:  delta + p*F  >  25")
print()
print(f"{'delta':>7}{'p*F needed':>13}{'p* at F=50':>13}   note")
print("-" * 62)
for d in [0.0, 5.0, 10.0, 15.0, 20.0, 24.0, 25.0]:
    need = max(0.0, 25.0 - d)
    ps = need / 50.0
    note = "no enforcement needed" if need == 0 else ""
    print(f"{d:7.0f}{need:13.1f}{ps:13.4f}   {note}")

print()
print("Cross-check: honesty must be the UNIQUE best response just above the threshold")
print("(at the threshold itself the fake ties, so strict inequality is required)")
for d, p in [(0.0, 0.51), (15.0, 0.21), (24.5, 0.02)]:
    w = best_response(p=p, F=50.0, delta=d)
    unique = len(w) == 1 and w[0]["action"] == "A_honest"
    print(f"  delta={d:5.1f}, p={p:.2f}, F=50 -> pF+delta={p*50+d:5.1f}  "
          f"{'HONEST unique' if unique else 'NOT unique'}")

print()
print("And at the threshold itself the two tie, as predicted:")
for d, p in [(0.0, 0.50), (15.0, 0.20), (25.0, 0.00)]:
    w = best_response(p=p, F=50.0, delta=d)
    names = " | ".join(r["action"] for r in w)
    print(f"  delta={d:5.1f}, p={p:.2f} -> pF+delta={p*50+d:5.1f}  {names}")

print("\nThe arithmetic and the simulation agree in both regimes.")

## 6. What this notebook does *not* show

This is the honest boundary, and it matches the boundary stated in the paper.

- **The premium is a parameter, not a belief.** Delta is assumed here. In reality a premium exists only if publishing a trace is *informative about type*. If fakers can publish traces as cheaply as honest producers, the signal pools, Delta falls to zero, and only enforcement remains. This is Spence's single-crossing condition, and making Delta endogenous is the next study.
- **No human behavior.** Neither threshold is a property of people. Q3 asks whether traceability actually reduces dishonesty, or whether easy checking produces moral licensing.
- **No real platform.** The coefficients are a teaching calibration, not estimates. Entry, reputation, repeated interaction and heterogeneous fines are omitted.
- **Detection is free.** The platform is not charged for spot-checking, so its own optimisation over p is out of scope.
- **Zero receiver surplus.** Mäkimattila et al. show the certifier can capture the surplus, so a premium does not flow to honest producers automatically; competition or rule design must direct it. This notebook assumes it does.

### The next test

Endogenise Delta: let buyers update on whether a trace was published, and test whether publication separates types or pools them. A feasible companion experiment crosses trace-file presence with verifier scrutiny depth, measuring producer choice, check depth and false confidence.

### Changes required in the deployed artifact

The Hugging Face game should be updated to the corrected law in Section 1 and extended to expose Delta as a third control, so a player can observe that a premium substitutes for enforcement.

---

### Sources

- Akerlof, G. A. (1970). The market for "lemons". *Quarterly Journal of Economics*, 84(3), 488-500.
- Spence, M. (1973). Job market signaling. *Quarterly Journal of Economics*, 87(3), 355-374.
- Mäkimattila, M., Shang, Y., & Shirakawa, R. (2025). The Design and Price of Certification. *ACM EC '25*, 273. doi:10.1145/3736252.3742528
- Shalvi, S., Eldar, O., & Bereby-Meyer, Y. (2012). Honesty requires time (and lack of justifications). *Psychological Science*, 23(10), 1264-1270.
- Buneman, P., Khanna, S., & Tan, W.-C. (2001). Why and where: A characterization of data provenance. *ICDT 2001*, LNCS 1973, 316-330.
- SLSA authors. Supply-chain Levels for Software Artifacts, specification v1.2. https://slsa.dev/

### AI-use disclosure

The research question, the payoff design and both sweeps are the author's. The author used an AI text assistant (DeepSeek, `deepseek-v4-flash`, 6 September 2026) for debugging the client-side JavaScript of the interactive game. Both corrections recorded at the top of this notebook - the sign error and the missing premium - were found and made by the author, who ran every check here and remains responsible for every claim.